# Experiment Runner
Speaker / Listener × Logit / Choice × 모델 선택

**사용법**: Config 만 수정하여 사용

In [1]:
import csv
from pathlib import Path
from random import sample
from variables import *
from scoring import score_candidates

In [5]:
# ── CONFIG ──────────────────────── (여기만 수정)
MODEL       = "qwen3-8b"   # "llama3" | "qwen3"
ROLE        = "speaker"  # "speaker" | "listener"
MODE        = "choice"    # "logit"   | "choice"
REPETITIONS = 10          # logit → 1 권장 (결정론적),  choice → 5~10
SHOT = "zero"  # "zero" | "one" | "few"
# ────────────────────────────────────────────────

print(f"model={MODEL}  role={ROLE}  mode={MODE}  reps={REPETITIONS}")

model=qwen3-8b  role=speaker  mode=choice  reps=10


In [6]:
# ── 모델 로드 ─────────────────────────────────────
from llama_cpp import Llama

llm = Llama.from_pretrained(
    **MODEL_CONFIGS[MODEL],
    n_ctx=2048,
    n_threads=4,
    logits_all=True,
)
print(f"로드 완료: {MODEL}")

llama_model_load_from_file_impl: using device MTL0 (Apple M2) (unknown id) - 10922 MiB free
llama_model_loader: loaded meta data with 28 key-value pairs and 399 tensors from /Users/eyun/.cache/huggingface/hub/models--Qwen--Qwen3-8B-GGUF/snapshots/7c41481f57cb95916b40956ab2f0b139b296d974/./Qwen3-8B-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen3 8B Awq Compatible Instruct
llama_model_loader: - kv   3:                           general.finetune str              = awq-compatible-Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen3
llama_model_loader: - kv   5:

로드 완료: qwen3-8b


In [7]:
# ── 실험 루프 ─────────────────────────────────────
base_prompt = Path(f"prompts/{ROLE}_{MODE}_{SHOT}.txt").read_text(encoding="utf-8").strip()

candidates = ADJ_CANDIDATES if ROLE == "speaker" else STATE_CANDIDATES
outer_loop = STATE_VAR      if ROLE == "speaker" else ADJECTIVES

rows  = []
total = len(SITUATIONS) * len(RELATIONSHIP_VAR) * len(outer_loop) * REPETITIONS
print(f"총 {total} rows 생성 예정")
count = 0

for situation, scenario_template in SITUATIONS.items():
    for rel in RELATIONSHIP_VAR:
        for outer in outer_loop:
            for _ in range(REPETITIONS):
                personA, personB = sample(NAME_VARIATIONS, 2)
                scenario = scenario_template.format(personA=personA, personB=personB)
                thing    = THING_KEYWORDS[situation]

                fmt = dict(
                    personA=personA, personB=personB,
                    relationship=rel, scenario=scenario, thing=thing,
                )
                if ROLE == "speaker":
                    fmt["state"] = outer
                else:
                    fmt["adjective"] = f" {outer}"   # leading space for tokenization

                prompt = base_prompt.format(**fmt)
                
                # for qwen3, enable the thinking mode
                if MODE == "choice" and "qwen3" in MODEL:
                    prompt += "\n/no_think"

                row = dict(
                    situation=situation, relationship=rel,
                    personA=personA, personB=personB,
                )
                row["state" if ROLE == "speaker" else "adjective"] = outer

                if MODE == "logit":
                    logits, probs, logprobs = score_candidates(llm, prompt, candidates)
                    for c in candidates:
                        k = c.strip()
                        row[f"logit_{k}"]   = logits[c]
                        row[f"prob_{k}"]    = probs[c]
                        row[f"logprob_{k}"] = logprobs[c]
                    row["pred_top1"] = max(candidates, key=lambda c: probs[c]).strip()
                else:
                    resp = llm(prompt, max_tokens=30, temperature=0.0, top_k=1, top_p=0.9)
                    row["response_text"] = resp["choices"][0]["text"].strip()

                rows.append(row)
                count += 1
                if count % 100 == 0:
                    label = f"state={outer}" if ROLE == "speaker" else f"adj={outer}"
                    print(f"  [{count}/{total}] {situation} | {rel} | {label}")

Path("results").mkdir(exist_ok=True)
out = Path(f"results/{ROLE}_{MODE}_{MODEL}_{SHOT}.csv")
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print(f"\n완료 → {out}  ({len(rows)} rows)")

총 1000 rows 생성 예정


llama_perf_context_print:        load time =       0.38 ms
llama_perf_context_print: prompt eval time =       0.07 ms /   148 tokens (    0.00 ms per token, 2084507.04 tokens per second)
llama_perf_context_print:        eval time =    2361.56 ms /    29 runs   (   81.43 ms per token,    12.28 tokens per second)
llama_perf_context_print:       total time =    7641.35 ms /   177 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remaining 91 prompt tokens to eval
llama_perf_context_print:        load time =       0.38 ms
llama_perf_context_print: prompt eval time =    2409.45 ms /    91 tokens (   26.48 ms per token,    37.77 tokens per second)
llama_perf_context_print:        eval time =    2584.99 ms /    29 runs   (   89.14 ms per token,    11.22 tokens per second)
llama_perf_context_print:       total time =    5009.42 ms /   120 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remaini

  [100/1000] Kuchen | Entfernte Kollegin | state=5


llama_perf_context_print:        load time =       0.38 ms
llama_perf_context_print: prompt eval time =    3541.33 ms /    89 tokens (   39.79 ms per token,    25.13 tokens per second)
llama_perf_context_print:        eval time =    2690.51 ms /    29 runs   (   92.78 ms per token,    10.78 tokens per second)
llama_perf_context_print:       total time =    6244.77 ms /   118 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remaining 101 prompt tokens to eval
llama_perf_context_print:        load time =       0.38 ms
llama_perf_context_print: prompt eval time =    4057.82 ms /   101 tokens (   40.18 ms per token,    24.89 tokens per second)
llama_perf_context_print:        eval time =    2623.83 ms /    29 runs   (   90.48 ms per token,    11.05 tokens per second)
llama_perf_context_print:       total time =    6695.08 ms /   130 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remainin

  [200/1000] Kuchen | Gefürchtete Chefin | state=5


...

  [800/1000] Theater | Gefürchtete Chefin | state=5


llama_perf_context_print:        load time =       0.38 ms
llama_perf_context_print: prompt eval time =    6624.24 ms /   100 tokens (   66.24 ms per token,    15.10 tokens per second)
llama_perf_context_print:        eval time =    3433.03 ms /    29 runs   (  118.38 ms per token,     8.45 tokens per second)
llama_perf_context_print:       total time =   10075.20 ms /   129 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remaining 104 prompt tokens to eval
llama_perf_context_print:        load time =       0.38 ms
llama_perf_context_print: prompt eval time =    6956.14 ms /   104 tokens (   66.89 ms per token,    14.95 tokens per second)
llama_perf_context_print:        eval time =    3299.23 ms /    29 runs   (  113.77 ms per token,     8.79 tokens per second)
llama_perf_context_print:       total time =   10291.56 ms /   133 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remainin

  [900/1000] Gitarre | Entfernte Kollegin | state=5


llama_perf_context_print:        load time =       0.38 ms
llama_perf_context_print: prompt eval time =    7905.83 ms /   101 tokens (   78.28 ms per token,    12.78 tokens per second)
llama_perf_context_print:        eval time =    3451.38 ms /    29 runs   (  119.01 ms per token,     8.40 tokens per second)
llama_perf_context_print:       total time =   11396.50 ms /   130 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remaining 100 prompt tokens to eval
llama_perf_context_print:        load time =       0.38 ms
llama_perf_context_print: prompt eval time =    7287.15 ms /   100 tokens (   72.87 ms per token,    13.72 tokens per second)
llama_perf_context_print:        eval time =    3423.34 ms /    29 runs   (  118.05 ms per token,     8.47 tokens per second)
llama_perf_context_print:       total time =   10727.55 ms /   129 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 45 prefix-match hit, remainin

  [1000/1000] Gitarre | Gefürchtete Chefin | state=5

완료 → results/speaker_choice_qwen3-8b_zero.csv  (1000 rows)


In [21]:
# ── 실험 루프 ─────────────────────────────────────
base_prompt = Path(f"prompts/{ROLE}_{MODE}_{SHOT}.txt").read_text(encoding="utf-8").strip()

candidates = ADJ_CANDIDATES if ROLE == "speaker" else STATE_CANDIDATES
outer_loop = STATE_VAR      if ROLE == "speaker" else ADJECTIVES

rows  = []
total = len(SITUATIONS) * len(RELATIONSHIP_VAR) * len(outer_loop) * REPETITIONS
print(f"총 {total} rows 생성 예정")
count = 0

for situation, scenario_template in SITUATIONS.items():
    for rel in RELATIONSHIP_VAR:
        for outer in outer_loop:
            for _ in range(REPETITIONS):
                personA, personB = sample(NAME_VARIATIONS, 2)
                scenario = scenario_template.format(personA=personA, personB=personB)
                thing    = THING_KEYWORDS[situation]

                fmt = dict(
                    personA=personA, personB=personB,
                    relationship=rel, scenario=scenario, thing=thing,
                )
                if ROLE == "speaker":
                    fmt["state"] = outer
                else:
                    fmt["adjective"] = f" {outer}"   # leading space for tokenization

                prompt = base_prompt.format(**fmt)

                if MODE == "choice" and "qwen3" in MODEL:
                    prompt += "\n/no_think"

                row = dict(
                    situation=situation, relationship=rel,
                    personA=personA, personB=personB,
                )
                row["state" if ROLE == "speaker" else "adjective"] = outer

                if MODE == "logit":
                    logits, probs, logprobs = score_candidates(llm, prompt, candidates)
                    for c in candidates:
                        k = c.strip()
                        row[f"logit_{k}"]   = logits[c]
                        row[f"prob_{k}"]    = probs[c]
                        row[f"logprob_{k}"] = logprobs[c]
                    row["pred_top1"] = max(candidates, key=lambda c: probs[c]).strip()
                else:
                    resp = llm(prompt, max_tokens=30, temperature=0.0, top_k=1, top_p=0.9)
                    row["response_text"] = resp["choices"][0]["text"].strip()

                rows.append(row)
                count += 1
                if count % 100 == 0:
                    label = f"state={outer}" if ROLE == "speaker" else f"adj={outer}"
                    print(f"  [{count}/{total}] {situation} | {rel} | {label}")

Path("results").mkdir(exist_ok=True)
out = Path(f"results/{ROLE}_{MODE}_{MODEL}_{SHOT}_test.csv")
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print(f"\n완료 → {out}  ({len(rows)} rows)")

총 100 rows 생성 예정


Llama.generate: 42 prefix-match hit, remaining 115 prompt tokens to eval
llama_perf_context_print:        load time =       1.88 ms
llama_perf_context_print: prompt eval time =    8434.01 ms /   116 tokens (   72.71 ms per token,    13.75 tokens per second)
llama_perf_context_print:        eval time =    2815.21 ms /    29 runs   (   97.08 ms per token,    10.30 tokens per second)
llama_perf_context_print:       total time =    9274.80 ms /   145 tokens
llama_perf_context_print:    graphs reused =         28
Llama.generate: 42 prefix-match hit, remaining 117 prompt tokens to eval
llama_perf_context_print:        load time =       1.88 ms
llama_perf_context_print: prompt eval time =    5166.75 ms /   117 tokens (   44.16 ms per token,    22.64 tokens per second)
llama_perf_context_print:        eval time =    2843.88 ms /    29 runs   (   98.06 ms per token,    10.20 tokens per second)
llama_perf_context_print:       total time =    8026.47 ms /   146 tokens
llama_perf_context_print:   

  [100/100] Gitarre | Gefürchtete Chefin | adj=schrecklich

완료 → results/listener_choice_qwen3-8b_zero_test.csv  (100 rows)
